# Ingesta incremental de mediciones a Bronze

Los CSV se mantienen en `recepcion/mediciones/lote_XX/`, `lote_correccion/` y `lote_reenvio/`. Auto Loader lee recursivamente desde la carpeta común y conserva el checkpoint.


In [36]:
import csv
import json
from pathlib import Path

from pyspark.sql import functions as F
from pyspark.sql.types import StringType, StructField, StructType

config_path = next(
    (parent / 'config' / 'bronze.json' for parent in (Path.cwd(), *Path.cwd().parents)
     if (parent / 'config' / 'bronze.json').is_file()), None
)
if config_path is None:
    raise FileNotFoundError('No se encuentra config/bronze.json desde el directorio del notebook')
config = json.loads(config_path.read_text(encoding='utf-8'))
INPUT_DIR = config['mediciones_input_dir']
TARGET_TABLE = config['tables']['mediciones']
CHECKPOINT = config['checkpoints']['mediciones']
expected_columns = list(config['input_schemas']['mediciones'])
assert all(dtype.lower() == 'string' for dtype in config['input_schemas']['mediciones'].values())
SOURCE_SCHEMA = StructType([StructField(name, StringType(), True) for name in expected_columns])
print('Entrada:', INPUT_DIR, '\nTabla:', TARGET_TABLE, '\nCheckpoint:', CHECKPOINT)


Entrada: /Volumes/workspace/lensdata_bronze/archivos/recepcion/mediciones 
Tabla: workspace.lensdata_bronze.mediciones 
Checkpoint: /Volumes/workspace/lensdata_bronze/archivos/checkpoints/bronze/mediciones


## Validación previa de archivos

Comprueba todas las cabeceras antes de iniciar la carga. Copia cada lote a su subcarpeta sin sobrescribir archivos ya procesados.


In [37]:
def csv_files(directory):
    for entry in dbutils.fs.ls(directory):
        if entry.path.endswith('/'):
            yield from csv_files(entry.path)
        elif entry.path.lower().endswith('.csv'):
            yield entry.path

files = sorted(csv_files(INPUT_DIR))
assert files, f'No hay archivos CSV en {INPUT_DIR}'
for file_path in files:
    first_line = spark.read.text(file_path).first()
    assert first_line is not None, f'Archivo vacío: {file_path}'
    actual_columns = next(csv.reader([first_line[0].lstrip('\ufeff')]))
    assert actual_columns == expected_columns, (
        f'Cabecera inesperada en {file_path}: {actual_columns}; esperada: {expected_columns}'
    )
print(f'Cabeceras verificadas: {len(files)}')
print(*files, sep='\n')


Cabeceras verificadas: 15
/Volumes/workspace/lensdata_bronze/archivos/recepcion/mediciones/lote_01/mediciones_2026-11-01.csv
/Volumes/workspace/lensdata_bronze/archivos/recepcion/mediciones/lote_02/mediciones_2026-11-02.csv
/Volumes/workspace/lensdata_bronze/archivos/recepcion/mediciones/lote_03/mediciones_2026-11-03.csv
/Volumes/workspace/lensdata_bronze/archivos/recepcion/mediciones/lote_03/mediciones_2026-11-04.csv
/Volumes/workspace/lensdata_bronze/archivos/recepcion/mediciones/lote_03/mediciones_2026-11-05.csv
/Volumes/workspace/lensdata_bronze/archivos/recepcion/mediciones/lote_03/mediciones_2026-11-06.csv
/Volumes/workspace/lensdata_bronze/archivos/recepcion/mediciones/lote_03/mediciones_2026-11-07.csv
/Volumes/workspace/lensdata_bronze/archivos/recepcion/mediciones/lote_03/mediciones_2026-11-08.csv
/Volumes/workspace/lensdata_bronze/archivos/recepcion/mediciones/lote_03/mediciones_2026-11-09.csv
/Volumes/workspace/lensdata_bronze/archivos/recepcion/mediciones/lote_03/mediciones

## Ingesta

Bronze conserva las ocho columnas CSV como cadenas y la ruta completa en `_source_file`. La limpieza, el tipado y la deduplicación pertenecen a Silver.


In [38]:
# La fuente y el checkpoint no cambian entre lotes. La tabla recibe filas nuevas en modo append.
mediciones = (spark.readStream.format('cloudFiles')
    .option('cloudFiles.format', 'csv')
    .option('header', 'true')
    .option('mode', 'FAILFAST')
    .option('ignoreLeadingWhiteSpace', 'false')
    .option('ignoreTrailingWhiteSpace', 'false')
    .option('recursiveFileLookup', 'true')
    .schema(SOURCE_SCHEMA)
    .load(INPUT_DIR)
    .withColumn('_source_file', F.col('_metadata.file_path')))

query = (mediciones.writeStream.format('delta')
    .option('checkpointLocation', CHECKPOINT)
    .trigger(availableNow=True)
    .toTable(TARGET_TABLE))
query.awaitTermination()
print('Carga finalizada. Ejecutar 03_validar_mediciones_bronze para comprobar la tabla.')


Carga finalizada. Ejecutar 03_validar_mediciones_bronze para comprobar la tabla.
